In [4]:
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pathlib import Path
import html

pptx_path = "[제안서] Ⅲ.3.1.7.1 통합로그_3p_v0.12_20260615.pptx"

out_dir = Path("elastic_log_html")
out_dir.mkdir(exist_ok=True)

prs = Presentation(pptx_path)

slide_w = prs.slide_width
slide_h = prs.slide_height


def emu_to_px(v):
    return int(v / 9525)


def iter_shapes(shapes):
    for shape in shapes:
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from iter_shapes(shape.shapes)
        else:
            yield shape


def clean_text(text):
    return (
        text.strip()
        .replace("\x0b", "\n")
        .replace("\r", "\n")
    )


for i, slide in enumerate(prs.slides, start=1):
    elements = []

    for shape in iter_shapes(slide.shapes):
        left = emu_to_px(shape.left)
        top = emu_to_px(shape.top)
        width = emu_to_px(shape.width)
        height = emu_to_px(shape.height)

        if left + width < 0 or top + height < 0:
            continue
        if left > emu_to_px(slide_w) or top > emu_to_px(slide_h):
            continue

        if not hasattr(shape, "text"):
            continue

        text = clean_text(shape.text)

        if not text:
            continue

        text_html = html.escape(text).replace("\n", "<br>")

        elements.append(f"""
        <div class="shape" style="
            left:{left}px;
            top:{top}px;
            width:{width}px;
            height:{height}px;
        ">
            {text_html}
        </div>
        """)

    slide_html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<style>
body {{
    margin: 0;
    background: #eee;
}}

.slide {{
    position: relative;
    width: {emu_to_px(slide_w)}px;
    height: {emu_to_px(slide_h)}px;
    background: white;
    overflow: hidden;
    font-family: Arial, 'Malgun Gothic', sans-serif;
}}

.shape {{
    position: absolute;
    box-sizing: border-box;
    font-size: 14px;
    line-height: 1.25;
    white-space: normal;
    overflow: hidden;
    padding: 2px;
}}
</style>
</head>
<body>
<section class="slide">
{''.join(elements)}
</section>
</body>
</html>
"""

    (out_dir / f"slide_{i:02d}.html").write_text(slide_html, encoding="utf-8")

print(f"완료: {out_dir.resolve()}")

완료: /root/proposal-automation/ppt-parser/elastic_log_html
